# Generative Networks

Recurrent Neural Networks (RNNs) aur unke gated cell variants jaise Long Short Term Memory cells (LSTMs) aur Gated Recurrent Units (GRUs) ne language modeling ke liye ek mechanism diya hai, yaani woh words ka order seekh sakte hain aur sequence mein agla word predict kar sakte hain. Isse hum RNNs ko **generative tasks** ke liye use kar sakte hain, jaise ordinary text generation, machine translation, aur even image captioning.

Pichhli unit mein humne jo RNN architecture discuss kiya tha, usme har RNN unit agla hidden state output ke roop mein produce karta tha. Haalaanki, hum har recurrent unit mein ek aur output add kar sakte hain, jo hamein ek **sequence** output karne dega (jo original sequence ki length ke barabar hogi). Iske alawa, hum aise RNN units bhi use kar sakte hain jo har step par input accept nahi karte, balki sirf ek initial state vector lete hain aur phir outputs ka ek sequence produce karte hain.

Is notebook mein, hum simple generative models par focus karenge jo hamein text generate karne mein help karte hain. Simplicity ke liye, chaliye ek **character-level network** banate hain, jo letter by letter text generate karta hai. Training ke dauran, hamein koi text corpus lena hoga aur use letter sequences mein split karna hoga.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

ds_train, ds_test = tfds.load('ag_news_subset').values()

## Character Vocabulary banana

Character-level generative network banane ke liye, hamein text ko words ke bajay individual letters mein split karna hoga. `TextVectorization` layer, jo humne pehle use ki thi, aisa nahi kar sakti, isliye hamare paas do options hain:

* Text ko manually load karein aur 'haath se' tokenization karein, jaisa [is official Keras example](https://keras.io/examples/generative/lstm_character_level_text_generation/) mein dikhaya gaya hai.
* Character-level tokenization ke liye `Tokenizer` class use karein.

Hum doosra option choose karenge. `Tokenizer` ko words mein tokenize karne ke liye bhi use kiya ja sakta hai, isliye koi aasaani se character-level se word-level tokenization par switch kar sakta hai.

Character-level tokenization karne ke liye, hamein `char_level=True` parameter pass karna hoga:

In [ ]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

tokenizer = keras.preprocessing.text.Tokenizer(char_level=True,lower=False)
tokenizer.fit_on_texts([x['title'].numpy().decode('utf-8') for x in ds_train])

Hum ek special token use karna chahte hain jo **end of sequence** ko darshaye, jise hum `<eos>` kahenge. Chaliye ise manually vocabulary mein add karte hain:

In [ ]:
eos_token = len(tokenizer.word_index)+1
tokenizer.word_index['<eos>'] = eos_token

vocab_size = eos_token + 1

In [ ]:
tokenizer.texts_to_sequences(['Hello, world!'])

## Headlines generate karne ke liye Generative RNN ko Train karna

Hum RNN ko news headlines generate karne ke liye is tarah train karenge. Har step par, hum ek headline lenge, jo RNN ko di jayegi, aur har input letter ke liye hum network se agla output letter generate karne ko kahenge:

!['HELLO' word ke RNN generation ka example dikhane wali image.](../../../../../lessons/5-NLP/17-GenerativeNetworks/images/rnn-generate.png)

Hamare sequence ke aakhri letter ke liye, hum network se `<eos>` token generate karne ko kahenge.

Yahan use ho rahe generative RNN ka main difference yeh hai ki hum RNN ke har step se output lenge, na ki sirf aakhri cell se. Isse RNN cell mein `return_sequences` parameter specify karke achieve kiya ja sakta hai.

Is tarah, training ke dauran, network ka input kisi length ke encoded characters ka sequence hoga, aur output usi length ka sequence hoga, lekin ek element se shift kiya hua aur `<eos>` se terminate kiya hua. Minibatch mein aise kai sequences honge, aur hamein saare sequences ko align karne ke liye **padding** use karni hogi.

Chaliye aisa function banate hain jo hamare liye dataset ko transform kare. Kyunki hum minibatch level par sequences ko pad karna chahte hain, hum pehle `.batch()` call karke dataset ko batch karenge, aur phir transformation karne ke liye use `map` karenge. Isliye, transformation function poore minibatch ko ek parameter ke roop mein lega:

In [ ]:
def title_batch(x):
    x = [t.numpy().decode('utf-8') for t in x]
    z = tokenizer.texts_to_sequences(x)
    z = tf.keras.preprocessing.sequence.pad_sequences(z)
    return tf.one_hot(z,vocab_size), tf.one_hot(tf.concat([z[:,1:],tf.constant(eos_token,shape=(len(z),1))],axis=1),vocab_size)

Yahan hum kuch important cheezein karte hain:
* Sabse pehle hum string tensor se actual text nikalte hain
* `text_to_sequences` strings ki list ko integer tensors ki list mein convert karta hai
* `pad_sequences` un tensors ko unki maximum length tak pad karta hai
* Aakhir mein hum saare characters ko one-hot encode karte hain, aur saath hi shifting aur `<eos>` add karne ka kaam bhi karte hain. Hum jaldi hi dekhenge ki hamein one-hot-encoded characters ki zaroorat kyun hai

Haalaanki, yeh function **Pythonic** hai, yaani ise Tensorflow ke computational graph mein automatically translate nahi kiya ja sakta. Agar hum is function ko seedhe `Dataset.map` function mein use karne ki koshish karenge, toh hamein errors milenge. Hamein is Pythonic call ko `py_function` wrapper use karke enclose karna hoga:

In [ ]:
def title_batch_fn(x):
    x = x['title']
    a,b = tf.py_function(title_batch,inp=[x],Tout=(tf.float32,tf.float32))
    return a,b

> **Note**: Pythonic aur Tensorflow transformation functions ke beech difference karna thoda complicated lag sakta hai, aur aap soch sakte hain ki hum dataset ko `fit` mein pass karne se pehle standard Python functions use karke transform kyun nahi karte. Haalaanki yeh definitely kiya ja sakta hai, `Dataset.map` use karne ka ek bada fayda hai, kyunki data transformation pipeline Tensorflow computational graph use karke execute hoti hai, jo GPU computation ka fayda uthati hai aur CPU/GPU ke beech data pass karne ki zaroorat ko kam karti hai.

Ab hum apna generator network bana sakte hain aur training shuru kar sakte hain. Ise kisi bhi recurrent cell par based kiya ja sakta hai jise humne pichhli unit mein discuss kiya tha (simple, LSTM ya GRU). Hamare example mein hum LSTM use karenge.

Kyunki network input ke roop mein characters leta hai, aur vocabulary size kaafi chhota hai, hamein embedding layer ki zaroorat nahi hai. One-hot-encoded input seedha LSTM cell mein ja sakta hai. Output layer ek `Dense` classifier hogi jo LSTM output ko one-hot-encoded token numbers mein convert kar degi.

Iske alawa, kyunki hum variable-length sequences ke saath kaam kar rahe hain, hum `Masking` layer use kar sakte hain taaki ek mask ban sake jo string ke padded hisse ko ignore kar de. Yeh strictly zaroori nahi hai, kyunki hamein `<eos>` token ke aage ki cheezon mein zyada interest nahi hai, lekin hum is layer type ke saath thoda experience paane ke liye ise use karenge. `input_shape` `(None, vocab_size)` hoga, jahan `None` variable length ke sequence ko indicate karta hai, aur output shape bhi `(None, vocab_size)` hogi, jaisa aap `summary` se dekh sakte hain:

In [ ]:
model = keras.models.Sequential([
    keras.layers.Masking(input_shape=(None,vocab_size)),
    keras.layers.LSTM(128,return_sequences=True),
    keras.layers.Dense(vocab_size,activation='softmax')
])

model.summary()
model.compile(loss='categorical_crossentropy')

model.fit(ds_train.batch(8).map(title_batch_fn))

## Output Generate karna

Ab jab humne model train kar liya hai, hum ise kuch output generate karne ke liye use karna chahte hain. Sabse pehle, hamein token numbers ki series se represent kiye gaye text ko decode karne ka ek tareeka chahiye. Iske liye, hum `tokenizer.sequences_to_texts` function use kar sakte hain; haalaanki, yeh character-level tokenization ke saath achhe se kaam nahi karta. Isliye, hum tokenizer se tokens ki ek dictionary (jise `word_index` kehte hain) lenge, ek reverse map banayenge, aur apna khud ka decoding function likhenge:

In [ ]:
reverse_map = {val:key for key, val in tokenizer.word_index.items()}

def decode(x):
    return ''.join([reverse_map[t] for t in x])

Ab, chaliye generation karte hain. Hum kisi string `start` se shuruaat karenge, use ek sequence `inp` mein encode karenge, aur phir har step par hum apne network ko call karenge taaki agla character predict ho sake.

Network ka output `out` ek vector hota hai jisme `vocab_size` elements hote hain, jo har token ki probabilities ko represent karte hain. Hum `argmax` use karke sabse likely token number dhoondh sakte hain. Uske baad hum is character ko generated tokens ki list mein add karte hain aur generation ka process continue rakhte hain. Is process mein ek character generate karne ki process ko `size` baar repeat kiya jaata hai taaki required number of characters generate ho sakein, aur jab `eos_token` mil jaata hai toh hum jaldi khatam kar dete hain.

In [ ]:
def generate(model,size=100,start='Today '):
        inp = tokenizer.texts_to_sequences([start])[0]
        chars = inp
        for i in range(size):
            out = model(tf.expand_dims(tf.one_hot(inp,vocab_size),0))[0][-1]
            nc = tf.argmax(out)
            if nc==eos_token:
                break
            chars.append(nc.numpy())
            inp = inp+[nc]
        return decode(chars)
    
generate(model)

## Training ke dauran Output ka Sample lena

Kyunki hamare paas *accuracy* jaisi koi useful metrics nahi hain, isliye yeh dekhne ka ek hi tareeka ki hamara model behtar ho raha hai, **training ke dauran generated string ka sample lena** hai. Isse karne ke liye, hum **callbacks** use karenge, yaani aise functions jinhe hum `fit` function mein pass kar sakte hain, aur jo training ke dauran periodically call kiye jaayenge.

In [ ]:
sampling_callback = keras.callbacks.LambdaCallback(
  on_epoch_end = lambda batch, logs: print(generate(model))
)

model.fit(ds_train.batch(8).map(title_batch_fn),callbacks=[sampling_callback],epochs=3)

Yeh example pehle se hi kaafi achha text generate karta hai, lekin ise kai tareekon se aur behtar banaya ja sakta hai:

* **Zyada text**. Humne apne task ke liye sirf headlines use ki hain, lekin aap poore text ke saath experiment karna chah sakte hain. Yaad rakhiye ki RNNs lambe sequences handle karne mein bahut achhe nahi hote, isliye unhe chhote sentences mein split karna ya hamesha kisi predefined value `num_chars` (jaise 256) ki fixed sequence length par training dena samajhdaari ho sakti hai. Aap upar diye gaye example ko aisi structure mein badalne ki koshish kar sakte hain, [official Keras tutorial](https://keras.io/examples/generative/lstm_character_level_text_generation/) ko inspiration ke roop mein use karte hue.

* **Multilayer LSTM**. LSTM cells ki 2 ya 3 layers try karna samajhdaari ho sakta hai. Jaisa humne pichhli unit mein mention kiya tha, LSTM ki har layer text se kuch patterns extract karti hai, aur character-level generator ke case mein hum expect kar sakte hain ki neeche wali LSTM layer letters extract karne ke liye responsible hogi, aur upar wali layers - words aur word combinations ke liye. Ise LSTM constructor mein layers ki number ka parameter pass karke aasaani se implement kiya ja sakta hai.

* Aap **GRU units** ke saath bhi experiment karna chah sakte hain aur dekh sakte hain ki kaun sa behtar perform karta hai, aur **alag-alag hidden layer sizes** ke saath bhi. Bahut badi hidden layer overfitting ka kaaran ban sakti hai (jaise network exact text seekh lega), aur chhota size achha result produce nahi kar sakta.

## Soft Text Generation aur Temperature

`generate` ki pichhli definition mein, hum hamesha us character ko agla character choose karte the jiski probability sabse zyada hoti thi. Iska result yeh hota tha ki text aksar baar-baar ek hi character sequence mein "cycle" karne lagta tha, jaise is example mein:
```
today of the second the company and a second the company ...
```

Haalaanki, agar hum agle character ke liye probability distribution dekhein, toh ho sakta hai ki kuch highest probabilities ke beech ka difference bahut bada na ho, jaise ek character ki probability 0.2 ho, aur doosre ki 0.19. Jaise, jab sequence '*play*' mein agla character dhoondha jaata hai, toh agla character barabar space ya **e** (jaise word *player* mein) ho sakta hai.

Isse yeh conclusion nikalta hai ki hamesha highest probability wala character choose karna "fair" nahi hai, kyunki doosra highest choose karna bhi hamein meaningful text ki taraf le ja sakta hai. Zyada samajhdaari yeh hogi ki network output dwara di gayi probability distribution se characters ko **sample** kiya jaaye.

Yeh sampling `np.multinomial` function use karke ki ja sakti hai, jo so-called **multinomial distribution** implement karta hai. Ek function jo is **soft** text generation ko implement karta hai, neeche define kiya gaya hai:

In [ ]:
def generate_soft(model,size=100,start='Today ',temperature=1.0):
        inp = tokenizer.texts_to_sequences([start])[0]
        chars = inp
        for i in range(size):
            out = model(tf.expand_dims(tf.one_hot(inp,vocab_size),0))[0][-1]
            probs = tf.exp(tf.math.log(out)/temperature).numpy().astype(np.float64)
            probs = probs/np.sum(probs)
            nc = np.argmax(np.random.multinomial(1,probs,1))
            if nc==eos_token:
                break
            chars.append(nc)
            inp = inp+[nc]
        return decode(chars)

words = ['Today ','On Sunday ','Moscow, ','President ','Little red riding hood ']
    
for i in [0.3,0.8,1.0,1.3,1.8]:
    print(f"\n--- Temperature = {i}")
    for j in range(5):
        print(generate_soft(model,size=300,start=words[j],temperature=i))

Humne ek aur parameter **temperature** introduce kiya hai, jo yeh indicate karne ke liye use hota hai ki hamein highest probability se kitni strongly chipke rehna chahiye. Agar temperature 1.0 hai, toh hum fair multinomial sampling karte hain, aur jab temperature infinity ki taraf jaata hai - saari probabilities barabar ho jaati hain, aur hum agla character randomly choose karte hain. Neeche diye gaye example mein hum dekh sakte hain ki jab hum temperature ko bahut zyada badha dete hain toh text meaningless ho jaata hai, aur jab yeh 0 ke kareeb hota hai toh yeh "cycled" hard-generated text jaisa dikhta hai.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->